# Can We Catch Dangerous Heart Rhythms During Surgery?
## Intraoperative Arrhythmia Detection from the ECG

Every anaesthesiologist has experienced that moment: you glance at the monitor and the heart rate tracing looks… *different*. The rhythm is irregular, or too fast, or there's a pattern you don't quite recognise. Is it dangerous? Should you treat it? Or is it just an artefact from the surgeon leaning on the patient?

**Perioperative arrhythmias are remarkably common.** Studies report that cardiac rhythm disturbances occur in up to 70% of patients undergoing non-cardiac surgery under general anaesthesia — most of them never documented, many of them never even noticed. The majority are benign: a few premature beats, a transient sinus bradycardia from a vagal stimulus. But some are not. Atrial fibrillation can precipitate haemodynamic collapse in a patient with aortic stenosis. A run of ventricular tachycardia can degenerate into cardiac arrest. Even sustained supraventricular tachycardia (SVT) can cause dangerous hypotension in a patient who is already volume-depleted and vasodilated from anaesthetic drugs.

The challenge is that **the operating room is a noisy environment** — both electrically and physiologically. Diathermy creates artefact. Patient movement, changes in ventilation, and surgical manipulation all affect the ECG. And you, the anaesthesiologist, are simultaneously managing the airway, titrating drugs, monitoring blood loss, and communicating with the surgical team. You cannot stare at the ECG trace for four hours straight.

So here is the question we explore in this chapter: **can we teach a computer to watch the ECG for us and flag the rhythms that matter?**

---

### Why does this matter clinically?

Perioperative arrhythmias are not just an academic curiosity. They are associated with:

- **Increased 30-day mortality** — particularly new-onset atrial fibrillation
- **Longer ICU and hospital stays** — even after controlling for surgical complexity
- **Perioperative stroke** — arrhythmias like AF are a major risk factor for thromboembolic events
- **Haemodynamic instability** — loss of atrial "kick" in AF or rapid rates in SVT reduce cardiac output precisely when the patient can least afford it

Despite this, most anaesthetic records capture heart rhythm as a single entry — "sinus rhythm" — taken at induction and never revisited. Continuous rhythm monitoring happens, but continuous rhythm *analysis* does not. The data is there, flowing through the monitor in real time. We simply do not use it.

This chapter shows you how to change that.

---

### What you will learn

By the end of this notebook, you will understand:

- **What the ECG signal actually looks like** as raw data — not the tidy textbook trace, but the messy, noisy, real-world waveform recorded during actual surgery
- **How to find heartbeats in the signal** — specifically, how to detect **R-peaks** (the tall, sharp spikes in each heartbeat) using signal processing
- **What RR intervals tell us about rhythm** — the time between consecutive heartbeats is the single most informative measurement for arrhythmia detection
- **How heart rate variability (HRV) captures rhythm patterns** — and why a heart that beats *too* regularly can be just as concerning as one that beats irregularly
- **How to classify common perioperative arrhythmias** — including atrial fibrillation (AF), supraventricular tachycardia (SVT), ventricular tachycardia (VT), bradycardia, and premature ectopic beats
- **How machine learning can automate rhythm classification** — turning continuous ECG streams into clinically actionable alerts

We will build everything from first principles, starting with a refresher on what the ECG waveform actually represents.

---

### A quick refresher: What are we looking at on the ECG?

You learned this in medical school, but let us revisit it through the lens of signal processing — because to teach a computer to read an ECG, we need to describe it in precise, mathematical terms.

Each heartbeat produces a characteristic electrical waveform with several components:

| Wave | What it represents | Typical duration |
|------|--------------------|-----------------|
| **P wave** | Atrial depolarisation — the atria contracting to push blood into the ventricles | 80–120 ms |
| **QRS complex** | Ventricular depolarisation — the ventricles contracting to eject blood | 80–120 ms |
| **T wave** | Ventricular repolarisation — the ventricles "resetting" electrically | 120–200 ms |

The **R-peak** is the tallest, sharpest deflection in the QRS complex. It is the easiest part of the ECG for a computer to detect — think of it as the "landmark" that says "a heartbeat happened here." Once we can reliably find every R-peak, we can measure the time between consecutive peaks. This time gap is called the **RR interval**, and it is the foundation of everything we do in this chapter.

Why? Because **arrhythmias are, at their core, disorders of timing.** A normal sinus rhythm produces RR intervals that are fairly regular (with subtle, healthy variation). Atrial fibrillation produces RR intervals that are chaotically irregular. Ventricular tachycardia produces RR intervals that are short and dangerously regular. Premature beats produce one short interval followed by a compensatory long one.

If we can accurately measure every RR interval in a surgical case, we have the raw material to detect most clinically important arrhythmias — without needing to analyse the full complexity of the ECG waveform morphology.

---

### What is heart rate variability, and why do we care?

**Heart rate variability (HRV)** is simply the variation in time between successive heartbeats. Your heart does *not* beat like a metronome — and that is a good thing.

In a healthy, awake patient, the heart rate fluctuates continuously. It speeds up slightly with each inspiration (the respiratory sinus arrhythmia you learned about in physiology) and slows down with expiration. This variability reflects the balance between the **sympathetic** ("fight or flight") and **parasympathetic** (vagal, "rest and digest") branches of the autonomic nervous system.

Under general anaesthesia, HRV changes dramatically. Volatile anaesthetics and opioids blunt autonomic reflexes. Positive-pressure ventilation alters the normal respiratory variation. The result is a heart that often beats more *regularly* than normal — which paradoxically tells us less about what is happening.

When we calculate HRV features from the RR interval series, we are really asking: **how much variation is there in beat-to-beat timing, and what patterns does that variation follow?** These features turn out to be powerful inputs for arrhythmia classification:

- **SDNN** — the standard deviation of all RR intervals (overall variability)
- **RMSSD** — the root mean square of successive RR differences (short-term, beat-to-beat variability — reflects vagal tone)
- **pNN50** — the percentage of consecutive RR intervals that differ by more than 50 ms (another marker of parasympathetic activity)
- **Coefficient of variation of RR** — SDNN normalised by the mean RR interval, allowing comparison across different heart rates

High irregularity (high RMSSD, high CoV) in the context of a normal or high heart rate? Think atrial fibrillation. Very low variability with a fast rate? Could be SVT or VT. A sudden spike in the beat-to-beat difference? Likely a premature ectopic beat.

---

### Our approach: from signal to diagnosis

We will work through this chapter in a logical sequence, building from simple to complex:

1. **Load and explore** intraoperative ECG data from VitalDB
2. **Visualise raw waveforms** — see what the ECG looks like as a time series of voltage values
3. **Detect R-peaks** — find every heartbeat in the signal
4. **Compute RR intervals and HRV features** — extract the timing information that encodes rhythm
5. **Classify rhythms** — use these features to identify normal sinus rhythm versus various arrhythmias
6. **Evaluate our detector** — how well does an automated system perform, and where does it fail?

**Dataset:** We use the VitalDB `tracks.csv` file, focusing on the **ECG_II** channel recorded by the SNUADC device — Lead II is the standard monitoring lead in the operating room because it provides the clearest view of the P wave and R-peak.

Let us begin.